# Notebook 01 — Data Profiling

**Project:** APAC Territory Planning  
**Objective:** Validate the synthetic dataset — schema, nulls, distributions, and early signals — before any analysis begins.

**Tables:** accounts · reps · assignments · opportunities · customers

In [1]:
import pandas as pd
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = "../data/raw"

In [2]:
accounts     = pd.read_csv(f"{DATA_DIR}/accounts.csv")
reps         = pd.read_csv(f"{DATA_DIR}/reps.csv")
assignments  = pd.read_csv(f"{DATA_DIR}/assignments.csv")
opportunities = pd.read_csv(f"{DATA_DIR}/opportunities.csv")
customers    = pd.read_csv(f"{DATA_DIR}/customers.csv")

tables = {
    "accounts":      accounts,
    "reps":          reps,
    "assignments":   assignments,
    "opportunities": opportunities,
    "customers":     customers,
}

for name, df in tables.items():
    print(f"{name:<15} {len(df):>6,} rows  {len(df.columns):>2} cols")

accounts         5,000 rows  10 cols
reps                20 rows   6 cols
assignments      5,000 rows   7 cols
opportunities    2,118 rows   8 cols
customers        1,055 rows   9 cols


## 1. Schema & Null Check

In [3]:
for name, df in tables.items():
    null_counts = df.isnull().sum()
    null_counts = null_counts[null_counts > 0]
    print(f"{'─' * 50}")
    print(f"  {name.upper()}")
    print(f"{'─' * 50}")
    for col, dtype in df.dtypes.items():
        nulls = null_counts.get(col, 0)
        null_str = f"  ← {nulls:,} nulls" if nulls > 0 else ""
        print(f"  {col:<25} {str(dtype):<12}{null_str}")
    print()

──────────────────────────────────────────────────
  ACCOUNTS
──────────────────────────────────────────────────
  account_id                int64       
  company_name              object      
  country                   object      
  subregion                 object      
  industry                  object      
  employee_band             object      
  estimated_arr             float64     
  segment                   object      
  is_customer               int64       
  account_status            object      

──────────────────────────────────────────────────
  REPS
──────────────────────────────────────────────────
  rep_id                    int64       
  rep_name                  object      
  subregion                 object      
  segment_focus             object      
  quota_usd                 float64     
  max_accounts              int64       

──────────────────────────────────────────────────
  ASSIGNMENTS
──────────────────────────────────────────────────
  as

## 2. Accounts Distribution

In [4]:
# Accounts by subregion
by_subregion = (
    accounts.groupby("subregion")
    .agg(n_accounts=("account_id", "count"),
         n_customers=("is_customer", "sum"),
         total_arr=("estimated_arr", "sum"))
    .assign(customer_rate=lambda x: x["n_customers"] / x["n_accounts"] * 100,
            avg_arr=lambda x: x["total_arr"] / x["n_accounts"])
    .sort_values("n_accounts", ascending=False)
)

print("ACCOUNTS BY SUBREGION")
print(f"{'Subregion':<18} {'Accounts':>9} {'Customers':>10} {'Cust%':>7} {'Avg ARR':>12} {'Total ARR':>14}")
print("─" * 75)
for subregion, row in by_subregion.iterrows():
    print(f"{subregion:<18} {row['n_accounts']:>9,} {row['n_customers']:>10,} "
          f"{row['customer_rate']:>6.1f}% {row['avg_arr']:>12,.0f} {row['total_arr']:>14,.0f}")

ACCOUNTS BY SUBREGION
Subregion           Accounts  Customers   Cust%      Avg ARR      Total ARR
───────────────────────────────────────────────────────────────────────────
SEA                  1,500.0      322.0   21.5%      346,708    520,062,200
Greater China        1,200.0      186.0   15.5%      493,902    592,682,300
North Asia           1,000.0      284.0   28.4%      344,434    344,434,100
India                  800.0       93.0   11.6%      154,963    123,970,700
ANZ                    500.0      170.0   34.0%      487,358    243,679,100


## 3. Whitespace & Coverage

In [5]:
coverage = (
    assignments
    .merge(accounts[["account_id", "subregion"]], on="account_id")
    .groupby(["subregion", "coverage_status"])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1),
            whitespace_pct=lambda x: x["Unassigned"] / x["total"] * 100)
    .sort_values("whitespace_pct", ascending=False)
)

print("COVERAGE BY SUBREGION")
print(f"{'Subregion':<18} {'Assigned':>9} {'Unassigned':>11} {'Whitespace%':>12}")
print("─" * 55)
for subregion, row in coverage.iterrows():
    print(f"{subregion:<18} {row['Assigned']:>9,} {row['Unassigned']:>11,} {row['whitespace_pct']:>11.1f}%")

COVERAGE BY SUBREGION
Subregion           Assigned  Unassigned  Whitespace%
───────────────────────────────────────────────────────
India                  570.0       230.0        28.7%
Greater China          879.0       321.0        26.8%
SEA                  1,348.0       152.0        10.1%
North Asia             952.0        48.0         4.8%
ANZ                    487.0        13.0         2.6%


## 4. Engagement Status

In [6]:
engagement = (
    assignments
    .merge(accounts[["account_id", "subregion"]], on="account_id")
    .groupby(["subregion", "engagement_status"])
    .size()
    .unstack(fill_value=0)
)

# Ensure consistent column order
for col in ["Active", "Warm", "Stale", "No Coverage"]:
    if col not in engagement.columns:
        engagement[col] = 0

engagement = engagement[["Active", "Warm", "Stale", "No Coverage"]]
engagement["total"] = engagement.sum(axis=1)

print("ENGAGEMENT BY SUBREGION")
print(f"{'Subregion':<18} {'Active':>7} {'Warm':>7} {'Stale':>7} {'No Cover':>9} {'Total':>7}")
print("─" * 60)
for subregion, row in engagement.iterrows():
    print(f"{subregion:<18} {row['Active']:>7,.0f} {row['Warm']:>7,.0f} "
          f"{row['Stale']:>7,.0f} {row['No Coverage']:>9,.0f} {row['total']:>7,.0f}")

ENGAGEMENT BY SUBREGION
Subregion           Active    Warm   Stale  No Cover   Total
────────────────────────────────────────────────────────────
ANZ                     25      51     411        13     500
Greater China           37      83     759       321   1,200
India                   26      34     510       230     800
North Asia              57     103     792        48   1,000
SEA                     66     125   1,157       152   1,500


## 5. Rep Capacity

In [7]:
rep_load = (
    assignments[assignments["coverage_status"] == "Assigned"]
    .groupby("rep_id")
    .size()
    .reset_index(name="assigned_accounts")
)

rep_summary = (
    reps.merge(rep_load, on="rep_id", how="left")
    .assign(assigned_accounts=lambda x: x["assigned_accounts"].fillna(0),
            load_pct=lambda x: x["assigned_accounts"] / x["max_accounts"] * 100)
    .sort_values("load_pct", ascending=False)
)

print("REP CAPACITY")
print(f"{'Rep':<22} {'Subregion':<16} {'Segment':<12} {'Quota':>10} {'Assigned':>9} {'Max':>5} {'Load%':>7}")
print("─" * 85)
for _, row in rep_summary.iterrows():
    print(f"{row['rep_name']:<22} {row['subregion']:<16} {row['segment_focus']:<12} "
          f"{row['quota_usd']:>10,.0f} {row['assigned_accounts']:>9,.0f} "
          f"{row['max_accounts']:>5} {row['load_pct']:>6.1f}%")

REP CAPACITY
Rep                    Subregion        Segment           Quota  Assigned   Max   Load%
─────────────────────────────────────────────────────────────────────────────────────
Douglas Young          North Asia       Enterprise    1,432,000       345    40  862.5%
Kiara Bell             North Asia       Enterprise    1,440,000       317    40  792.5%
Raven Fuller           ANZ              Enterprise    1,200,000       251    40  627.5%
Sandy Taylor           ANZ              Enterprise    1,200,000       236    40  590.0%
David Mitchell         SEA              Enterprise    1,200,000       221    40  552.5%
Brian Shannon          SEA              Enterprise    1,200,000       210    40  525.0%
Christopher Carter     North Asia       Mid-Market      809,000       290    80  362.5%
Julie Cole             Greater China    Mid-Market      700,000       232    80  290.0%
Johnny Woods           SEA              Mid-Market      700,000       225    80  281.2%
Kenneth Edwards      

## 6. Pipeline & Revenue Snapshot

In [8]:
# Pipeline by stage
pipeline = (
    opportunities
    .groupby("stage")
    .agg(n_opps=("opportunity_id", "count"),
         total_arr=("arr_potential", "sum"))
    .sort_values("total_arr", ascending=False)
)

print("PIPELINE BY STAGE")
print(f"{'Stage':<15} {'Opps':>6} {'Total ARR':>14}")
print("─" * 38)
for stage, row in pipeline.iterrows():
    print(f"{stage:<15} {row['n_opps']:>6,} {row['total_arr']:>14,.0f}")

print()

# Customer ARR by subregion
cust_arr = (
    customers
    .merge(accounts[["account_id", "subregion"]], on="account_id")
    .groupby("subregion")
    .agg(n_customers=("customer_id", "count"),
         total_arr=("arr", "sum"),
         avg_arr=("arr", "mean"))
    .sort_values("total_arr", ascending=False)
)

print("CUSTOMER ARR BY SUBREGION")
print(f"{'Subregion':<18} {'Customers':>10} {'Total ARR':>14} {'Avg ARR':>12}")
print("─" * 57)
for subregion, row in cust_arr.iterrows():
    print(f"{subregion:<18} {row['n_customers']:>10,} {row['total_arr']:>14,.0f} {row['avg_arr']:>12,.0f}")

PIPELINE BY STAGE
Stage             Opps      Total ARR
──────────────────────────────────────
Qualified        448.0    147,736,000
Demo             428.0    130,043,400
Closed Won       263.0    100,926,100
Proposal         340.0     95,530,200
Prospecting      300.0     93,551,000
Closed Lost      148.0     63,856,000
Negotiation      191.0     63,762,100

CUSTOMER ARR BY SUBREGION
Subregion           Customers      Total ARR      Avg ARR
─────────────────────────────────────────────────────────
SEA                     322.0     97,039,900      301,366
Greater China           186.0     96,946,500      521,218
North Asia              284.0     87,248,400      307,213
ANZ                     170.0     86,253,300      507,372
India                    93.0     13,093,600      140,791


## Key Findings

1. **Whitespace:** India (28.7%) and Greater China (26.8%) have the most unassigned accounts — highest upside
2. **Rep overload:** All Enterprise reps are at 500-860% capacity — account assignments don't reflect max_accounts limits
3. **India under-penetration:** Lowest customer rate (11.6%), lowest avg ARR ($140K), highest SMB mix
4. **ANZ paradox:** Most covered territory but reps still overloaded — needs redistribution not headcount
5. **Greater China:** Largest avg ARR ($521K) but lowest customer rate after India — biggest revenue upside per account